Please, change the type of execution time to "T4 GPU".

In [ ]:
!pip install -q accelerate==0.24.1 peft==0.6.2 bitsandbytes==0.41.2 transformers==4.35.0 datasets rouge einops loguru

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.4/261.4 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.7/174.7 kB 21.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.6/92.6 MB 10.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 107.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 521.2/521.2 kB 53.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 47.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 12.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 14.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 41.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.4/520.4 kB 38.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import sys
import torch
import torch.nn as nn
from accelerate import Accelerator #https://github.com/huggingface/accelerate
# Accelerator is a library that enables the same PyTorch code
# to be run across any distributed configuration by adding just four lines of code
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup, #https://huggingface.co/docs/transformers/main_classes/optimizer_schedules
    # ~NoamOpt but all linear
    set_seed,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset
import typer
from tqdm import tqdm
from loguru import logger
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
from rouge import Rouge

In [ ]:
from google.colab import drive
# Connect to drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
PROJ_ROOT = "/content/drive/Shareddrives/LLMs_Seminar/FinetuneLLMs/"
MODEL_NAME = "microsoft/phi-1_5" #model we are going to finetune

## Memory Trace and Stream Logger

In [ ]:
import torch
import gc
import psutil
import threading


def b2mb(x):
    '''
    Convert bytes to MB
    '''
    return int(x / 2**20)


class TorchTracemalloc:
    def __enter__(self):
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_max_memory_allocated()  # reset the peak gauge to zero
        self.begin = torch.cuda.memory_allocated()
        self.process = psutil.Process()

        self.cpu_begin = self.cpu_mem_used()
        self.peak_monitoring = True
        peak_monitor_thread = threading.Thread(target=self.peak_monitor_func)
        peak_monitor_thread.daemon = True
        peak_monitor_thread.start()
        return self

    def cpu_mem_used(self):
        """get resident set size memory for the current process"""
        return self.process.memory_info().rss

    def peak_monitor_func(self):
        self.cpu_peak = -1

        while True:
            self.cpu_peak = max(self.cpu_mem_used(), self.cpu_peak)

            # can't sleep or will not catch the peak right (this comment is here on purpose)
            # time.sleep(0.001) # 1msec

            if not self.peak_monitoring:
                break

    def __exit__(self, *exc):
        self.peak_monitoring = False

        gc.collect()
        torch.cuda.empty_cache()
        self.end = torch.cuda.memory_allocated()
        self.peak = torch.cuda.max_memory_allocated()
        self.used = b2mb(self.end - self.begin)
        self.peaked = b2mb(self.peak - self.begin)

        self.cpu_end = self.cpu_mem_used()
        self.cpu_used = b2mb(self.cpu_end - self.cpu_begin)
        self.cpu_peaked = b2mb(self.cpu_peak - self.cpu_begin)


In [ ]:
class StreamToLogger:
    def __init__(self, level="INFO"):
        self._level = level

    def write(self, buffer):
        print("writing")
        for line in buffer.rstrip().splitlines():
            logger.opt(depth=1).log(self._level, line.rstrip())

    def flush(self):
        pass

# Create dir to save weights

In [ ]:
logger.remove()
logger.add(sys.__stdout__)
timestamp = datetime.now().strftime("%m_%d_%Y_%H_%M_%S")
cli = typer.Typer()
OUTPUT_DIR = f"{PROJ_ROOT}/{MODEL_NAME.split('/')[-1]}/{timestamp}/"

CHECKPOINTS_ROOT_DIR = f"{PROJ_ROOT}/{MODEL_NAME.split('/')[-1]}/{timestamp}/checkpoints"
os.makedirs(CHECKPOINTS_ROOT_DIR, exist_ok=True)
logger.add(sink=os.path.join(OUTPUT_DIR, "log.log"))

2

In [ ]:
#just checking
print(OUTPUT_DIR)
print(CHECKPOINTS_ROOT_DIR)

/content/drive/Shareddrives/LLMs_Seminar/FinetuneLLMs//phi-1_5/11_22_2023_18_02_04/
/content/drive/Shareddrives/LLMs_Seminar/FinetuneLLMs//phi-1_5/11_22_2023_18_02_04/checkpoints


In [ ]:
# plot loss
def plot_history(history, n_epochs):
    legend = []
    for k, v in history.items():
      downs = int(len(v)/n_epochs)
      plt.plot(v[::downs])
      legend.append(k)
    plt.title('Loss')
    plt.ylabel('Loss')
    plt.xlabel('epoch')
    plt.legend(legend, loc='upper left')
    plt.savefig(os.path.join(OUTPUT_DIR, "loss.png"))
    plt.show()

# Preprocess Dataset

In [ ]:
accelerator = Accelerator()


def preproc_data(
    tokenizer: object,
    dataset: object,
    batch_size: int=8,
    max_length: int = 256,
):
   ### DATA PRE-PROCESSING

    # Split dataset
    dataset_train_val_split = dataset["train"].train_test_split(test_size=0.2)
    dataset["validation"] = dataset_train_val_split["test"]
    dataset["train"] = dataset_train_val_split["train"]

    #Pre-process function to tokenize and pad input and output sequences
    def preprocess_function(examples, padding="max_length"):
        #data = [question + label for question, label in zip(examples["query"], examples["target"])]
        model_inputs = tokenizer(
            examples["query"], examples["target"] #this depends on the field's names of the dataset you chose
            #max_length=max_length,
            #padding=padding,
            #truncation=True,
        )
        return model_inputs


    # Run all of the dataset preprocessing on the CPU main process
    with accelerator.main_process_first():
        # I think this dataset.map() allows for augmentation and pre-processing on the fly
        tokenized_datasets = dataset.map(
            preprocess_function,
            batched=True,
            num_proc=1,
            remove_columns=dataset["train"].column_names,
            load_from_cache_file=False,
            desc="Tokenizing dataset...",
        )
    #The primary purpose of map() is to speed up processing functions. It allows you to apply
    # a processing function to each example in a dataset, independently or in batches. This function
    # can even create new rows and columns.

    accelerator.wait_for_everyone()

    accelerator.print("End of Tokenizing") # instead of just "print", to print once by process

    train_dataset = tokenized_datasets["train"]
    eval_dataset = tokenized_datasets["validation"]

    ### Data collators are objects that will form a batch by using a list of dataset elements as input.
    # They also perform some pre-processing, as padding or masking (this latter is done by the DataCollatorForLanguageModelling)
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    ) #pads, creates the labels

    train_dataloader = DataLoader(
        train_dataset, #tokenized train dataset (by preprocess function)
        shuffle=True,
        collate_fn=data_collator,
        batch_size=batch_size,
        pin_memory=True,
    )
    eval_dataloader = DataLoader(
        eval_dataset,
        collate_fn=data_collator,
        batch_size=batch_size,
        pin_memory=True,
    )

    return train_dataloader, eval_dataloader


# Finetuning Function

In [ ]:
def train(
    model: object,
    train_dataloader: object,
    eval_dataloader: object,
    batch_size: int = 2,
    max_length: int = 256, #max sequence length
    num_epochs: int = 1,
    lr: float = 1e-4,
    gradient_accumulation_steps: int = 1,
    eval_step_num: int = 500, # when to evaluate and save the model
) -> None:

    loss_history = {"train": [], "val": []} #list of epoch losses during training
    global_step = 1 # Training steps

    logger.info(f"num_epochs: {num_epochs}")
    logger.info(f"batch size:{batch_size}, max_length: {max_length}")


    ###         TRAINING MODEL         ###


    # optimizer
    optimizer = torch.optim.AdamW(
        params=model.parameters(),
        lr=lr,
    )
    lr_scheduler = get_linear_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps= int(0.3*(len(train_dataloader) * num_epochs)),
        num_training_steps=(len(train_dataloader) * num_epochs), #no of batches * no of epochs
    )



    (
        model,
        train_dataloader,
        eval_dataloader,
        optimizer,
        lr_scheduler,
    ) = accelerator.prepare(
        model, train_dataloader, eval_dataloader, optimizer, lr_scheduler
    )



    ### EVALUATION LOOP Function
    def evaluation(model, step: int) -> None:
        logger.info(f"now evaluating model at step {step}")
        model.eval()
        eval_loss = 0
        with TorchTracemalloc() as tracemalloc:
            for _step, batch in enumerate(tqdm(eval_dataloader)):
                with torch.no_grad():
                    outputs = accelerator.unwrap_model(model)(
                        **batch,
                        # synced_gpus=is_ds_zero_3
                    )
                loss = outputs.loss #loss is already in model's architechture: CausalLMLoss
                                    #CrossEntropyLoss in this case
                eval_loss += loss.detach().float()

            epoch_eval_loss = eval_loss/len(eval_dataloader)
            loss_history["val"].append(epoch_eval_loss.item())
            logger.critical(
                f"eval_loss at global step {global_step}: {epoch_eval_loss}"
            )

        # Printing the GPU memory usage details such as allocated memory, peak memory, and total memory usage
        accelerator.print(
            "GPU Memory before entering the eval : {}".format(b2mb(tracemalloc.begin))
        )
        accelerator.print(
            "GPU Memory consumed at the end of the eval (end-begin): {}".format(
                tracemalloc.used
            )
        )
        accelerator.print(
            "GPU Peak Memory consumed during the eval (max-begin): {}".format(
                tracemalloc.peaked
            )
        )
        accelerator.print(
            "GPU Total Peak Memory consumed during the eval (max): {}".format(
                tracemalloc.peaked + b2mb(tracemalloc.begin)
            )
        )

        accelerator.print(
            "CPU Memory before entering the eval : {}".format(
                b2mb(tracemalloc.cpu_begin)
            )
        )
        accelerator.print(
            "CPU Memory consumed at the end of the eval (end-begin): {}".format(
                tracemalloc.cpu_used
            )
        )
        accelerator.print(
            "CPU Peak Memory consumed during the eval (max-begin): {}".format(
                tracemalloc.cpu_peaked
            )
        )
        accelerator.print(
            "CPU Total Peak Memory consumed during the eval (max): {}".format(
                tracemalloc.cpu_peaked + b2mb(tracemalloc.cpu_begin)
            )
        )

    ### Save model weights
    def dump_model(model, step: int) -> None:
        accelerator.print("SAVING MODEL")
        logger.info(f"dumping model at step {step}")
        model.save_pretrained(
            f"{CHECKPOINTS_ROOT_DIR}/{MODEL_NAME.split('/')[-1]}-finetuned-model-step-{step}"
        )


    ### TRAINING LOOP
    accelerator.print(f"NUMBER of BATCHES PER EPOCH - training: {len(train_dataloader)}; eval: {len(eval_dataloader)}")
    print(model.device)
    for epoch in range(num_epochs):
        with TorchTracemalloc() as tracemalloc:
            model.train()
            total_loss = 0
            for step, batch in enumerate(tqdm(train_dataloader)):
                outputs = model(**batch)
                # print(outputs.keys()) = 'loss', 'logits'

                loss = outputs.loss
                #loss = my_loss(batch['labels'], outputs.logits)
                #print("Compare to check if it's correct")
                # print(loss.item(), outputs.loss.item())

                loss = loss / gradient_accumulation_steps
                total_loss += loss.detach().float()
                accelerator.backward(loss)

                if step % gradient_accumulation_steps == 0:
                    #Divide gradients / gradient_accumulation_steps, so they won't be huge?
                    #for param in model.parameters():
                    # param.grad /= gradient_accumulation_steps
                    optimizer.step()
                    lr_scheduler.step()
                    optimizer.zero_grad()
                    model.zero_grad()

                global_step += 1
                if global_step % eval_step_num == 0:
                    accelerator.print("ENTER EVAL STEP")
                    evaluation(model=model, step=global_step)
                    dump_model(model=model, step=global_step)
                    model.train()



        # Printing the GPU memory usage details such as allocated memory, peak memory, and total memory usage
        accelerator.print(
            "GPU Memory before entering the train : {}".format(b2mb(tracemalloc.begin))
        )
        accelerator.print(
            "GPU Memory consumed at the end of the train (end-begin): {}".format(
                tracemalloc.used
            )
        )
        accelerator.print(
            "GPU Peak Memory consumed during the train (max-begin): {}".format(
                tracemalloc.peaked
            )
        )
        accelerator.print(
            "GPU Total Peak Memory consumed during the train (max): {}".format(
                tracemalloc.peaked + b2mb(tracemalloc.begin)
            )
        )

        accelerator.print(
            "CPU Memory before entering the train : {}".format(
                b2mb(tracemalloc.cpu_begin)
            )
        )
        accelerator.print(
            "CPU Memory consumed at the end of the train (end-begin): {}".format(
                tracemalloc.cpu_used
            )
        )
        accelerator.print(
            "CPU Peak Memory consumed during the train (max-begin): {}".format(
                tracemalloc.cpu_peaked
            )
        )
        accelerator.print(
            "CPU Total Peak Memory consumed during the train (max): {}".format(
                tracemalloc.cpu_peaked + b2mb(tracemalloc.cpu_begin)
            )
        )

        # End of epoch
        train_epoch_loss = total_loss / len(train_dataloader)
        train_ppl = torch.exp(train_epoch_loss).item()
        train_epoch_loss = train_epoch_loss.item()
        loss_history["train"].append(train_epoch_loss)
        logger.critical(
                f"train_loss at global step {global_step}: {train_epoch_loss}"
            )
        accelerator.print(f"{epoch=}: {train_ppl=} {train_epoch_loss=}")
        evaluation(model=model, step=global_step)
        dump_model(model=model, step=global_step)

    # Save final model
    dump_model(model=model, step=global_step)

    return loss_history

## Load Model

1. Quantize the model weights first, as colab doensn't have enough GPU VRAM to finetune a model without this.

(https://huggingface.co/docs/optimum/concept_guides/quantization)
Quantization is a technique to reduce the computational and memory costs of running inference by representing the weights and activations with low-precision data types like 8-bit integer (int8) instead of the usual 32-bit floating point (float32).

Reducing the number of bits means the resulting model requires less memory storage, consumes less energy (in theory), and operations like matrix multiplication can be performed much faster with integer arithmetic.

In [ ]:
################################################################################
# bitsandbytes parameters
# details about this at: https://colab.research.google.com/drive/1ge2F1QSK8Q7h0hn3YKuBCOAS0bK8E0wf
################################################################################

# Activate 4-bit precision base model loading
use_4bit = True

# Compute dtype for 4-bit base models
#change the dtype that will be used during computation. For example, hidden
#states could be in float32 but computation can be set to bf16 for speedups.
#By default, the compute dtype is set to float32
bnb_4bit_compute_dtype = "float16"
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

# Quantization type (fp4 or nf4)
#The 4bit integration comes with 2 different quantization types: FP4 and NF4.
#The NF4 dtype stands for Normal Float 4 and is introduced in the QLoRA paper
bnb_4bit_quant_type = "nf4"

# Activate nested quantization for 4-bit base models (double quantization)
#This they don't explain but they say it accelerates a lot
use_nested_quant = False

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit, #load the model in 4bit
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

In [ ]:
### Loading Model from Hugging Face (the class use here depends on the model you're loading)
# AutoModelForCausalLM -> with a causal language modeling head; predicts the next token
base_model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=MODEL_NAME,
    quantization_config=bnb_config,
    trust_remote_code=True, #needed for phi1.5 model
)

A new version of the following files was downloaded from https://huggingface.co/microsoft/phi-1_5:
- configuration_phi.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


A new version of the following files was downloaded from https://huggingface.co/microsoft/phi-1_5:
- modeling_phi.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


In [ ]:
base_model

PhiForCausalLM(
  (transformer): PhiModel(
    (embd): Embedding(
      (wte): Embedding(51200, 2048)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (h): ModuleList(
      (0-23): 24 x ParallelBlock(
        (ln): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (resid_dropout): Dropout(p=0.0, inplace=False)
        (mixer): MHA(
          (rotary_emb): RotaryEmbedding()
          (Wqkv): Linear4bit(in_features=2048, out_features=6144, bias=True)
          (out_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
          (inner_attn): SelfAttention(
            (drop): Dropout(p=0.0, inplace=False)
          )
          (inner_cross_attn): CrossAttention(
            (drop): Dropout(p=0.0, inplace=False)
          )
        )
        (mlp): MLP(
          (fc1): Linear4bit(in_features=2048, out_features=8192, bias=True)
          (fc2): Linear4bit(in_features=8192, out_features=2048, bias=True)
          (act): NewGELUActivation()
        )
      )
 

## LoRA finetuning configuration

In [ ]:
# https://huggingface.co/docs/peft/conceptual_guides/lora#lora-examples
peft_config = LoraConfig(
    r=8, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules= ['Wqkv'], #['Wqkv', 'out_proj', 'fc1', 'fc2']
    task_type="CAUSAL_LM" #layers_to_transform= [19, 20, 21, 22, 23],
) #larger the r, more parameters your learning

# Wrap the base model with get_peft_model() to get a trainable PeftModel.
model = get_peft_model(base_model, peft_config).to(base_model.device) #otherwise lora weight will be in cpu
model.print_trainable_parameters()

trainable params: 1,572,864 || all params: 1,419,843,584 || trainable%: 0.11077727277316766


In [ ]:
#for i in model.named_parameters():
#    if "lora" in i[0]:
#        print(f"{i[0]} -> {i[1].device}")

## Load Tokenizer

In [ ]:
## Loading Tokenizer from Hugging Face (the class use here depends on the model you're loading)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True) #trust_remote_code=True only for phi-1.5
print(tokenizer.pad_token)
tokenizer.pad_token = tokenizer.eos_token #eos_token = an end of sentence token
print(tokenizer.pad_token)

None
<|endoftext|>


## Load Dataset

In [ ]:
# Loading Dataset from Hugging Face
dataset_name = "ContextualAI/trivia_qa"

dataset = load_dataset(
                dataset_name
                )

Extracting data files:   0%|          | 0/3 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/78785 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11313 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/8837 [00:00<?, ? examples/s]

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['target', 'query', 'gold_generation'],
        num_rows: 78785
    })
    test: Dataset({
        features: ['target', 'query', 'gold_generation'],
        num_rows: 11313
    })
    dev: Dataset({
        features: ['target', 'query', 'gold_generation'],
        num_rows: 8837
    })
})

#### Understand DataCollatorforLanguageModelling

https://towardsdatascience.com/data-collators-in-huggingface-a0c76db798d2

In [ ]:
text = [ question + label for question, label in zip(dataset["train"]["query"][0:2], dataset["train"]["target"][0:2])]
print(text)
data = [tokenizer(t) for t in text]
print(data)

['Who was President when the first Peanuts cartoon was published?Harry Truman', 'Which American-born Sinclair won the Nobel Prize for Literature in 1930?Sinclair Lewis']
[{'input_ids': [8241, 373, 1992, 618, 262, 717, 2631, 37555, 16251, 373, 3199, 30, 18308, 33027], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}, {'input_ids': [13828, 1605, 12, 6286, 34927, 1839, 262, 20715, 15895, 329, 33818, 287, 15533, 30, 46200, 27659, 10174], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}]


In [ ]:
# This does the same thing as before, but instead of outputing a list, it agregates the two sequences
# In a tokenized batch
data2 = tokenizer(dataset["train"]["query"][0:2], dataset["train"]["target"][0:2])
data2

#for i, seq in enumerate(data):
#    print(i)
#    if seq['input_ids'] == data2['input_ids'][i]:
#        print("same shit")

{'input_ids': [[8241, 373, 1992, 618, 262, 717, 2631, 37555, 16251, 373, 3199, 30, 18308, 33027], [13828, 1605, 12, 6286, 34927, 1839, 262, 20715, 15895, 329, 33818, 287, 15533, 30, 46200, 27659, 10174]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

In [ ]:
collate_fn = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
dataloader = torch.utils.data.DataLoader(data, collate_fn=collate_fn, batch_size=2)

for batch in dataloader:
    print("----------------------------------")
    print(batch["input_ids"].shape, batch["attention_mask"].shape, batch["labels"].shape)
    print(batch)

## DataCollator pads up to the maximum sequence length in the batch.
## Attention mask has 0's for the padding tokens (id: 50256)
## Creates labels that are equal to the Inputs, cause in CLM, the model has to predict the next token
## -100 id in Labels is for the CrossEntropyLoss(ignore_idx=-100) -> ignores padding tokens

You're using a CodeGenTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


----------------------------------
torch.Size([2, 17]) torch.Size([2, 17]) torch.Size([2, 17])
{'input_ids': tensor([[ 8241,   373,  1992,   618,   262,   717,  2631, 37555, 16251,   373,
          3199,    30, 18308, 33027, 50256, 50256, 50256],
        [13828,  1605,    12,  6286, 34927,  1839,   262, 20715, 15895,   329,
         33818,   287, 15533,    30, 46200, 27659, 10174]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'labels': tensor([[ 8241,   373,  1992,   618,   262,   717,  2631, 37555, 16251,   373,
          3199,    30, 18308, 33027,  -100,  -100,  -100],
        [13828,  1605,    12,  6286, 34927,  1839,   262, 20715, 15895,   329,
         33818,   287, 15533,    30, 46200, 27659, 10174]])}


In [ ]:
tokenizer.decode(30), tokenizer.decode(50256)

('?', '<|endoftext|>')

## Finetune Model

In [ ]:
## DEFINE PARAMETERS
batch_size = 8
gradient_accumulation_steps = 1 #Gradient accumulation is a technique where you
        #can train on bigger batch sizes than your machine would normally be able
        #to fit into memory. This is done by accumulating gradients over several batches,
        #and only stepping the optimizer after a certain number of batches have been performed.
lr = 1e-4
num_epochs = 10
seed = 42
max_length = 128 ## NOT USING IT: cause the dataset sequences are not that large and the DataCollator in use
# pads each batch to the longest sequence length. This saves up memory
eval_step_num = 6000 # when to evaluate and save the model

set_seed(seed)

In [ ]:
#Pre-process datadet (split train dataset, padd, tokenize and create the dataloaders)
train_dataloader, eval_dataloader = preproc_data(tokenizer, dataset, batch_size, max_length)

Tokenizing dataset...:   0%|          | 0/63028 [00:00<?, ? examples/s]

Tokenizing dataset...:   0%|          | 0/11313 [00:00<?, ? examples/s]

Tokenizing dataset...:   0%|          | 0/8837 [00:00<?, ? examples/s]

Tokenizing dataset...:   0%|          | 0/15757 [00:00<?, ? examples/s]

End of Tokenizing


In [ ]:
i = 0
for batch in train_dataloader:
    print(batch['input_ids'][0:2], batch['labels'][0:2], batch['attention_mask'][0:2])
    print(batch['input_ids'].shape, batch['labels'].shape)
    print("\n ***** Decoded 1st two seqs of Batch ***")
    for j in range(2):
        print(tokenizer.decode(batch['input_ids'][j], skip_special_tokens=True))
        print("------------------------------")
        labels = batch['labels'][j]
        labels[labels == -100] = tokenizer.pad_token_id
        print(tokenizer.decode(labels, skip_special_tokens=True))
        print("*********************************")
    if i == 1:
        break
    i += 1

tensor([[ 7003,   407,   262,  1181,  3139,    11,   543,   318,   262,  4387,
          1748,   287, 18329,    30,    46,    76, 12236, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256],
        [ 2437,   867, 27198,   389,   319,   262, 12389,   286,   383, 43330,
           286, 14734,   287,   968,  1971,  4916,    30, 31334, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['target', 'query', 'gold_generation'],
        num_rows: 63028
    })
    test: Dataset({
        features: ['target', 'query', 'gold_generation'],
        num_rows: 11313
    })
    dev: Dataset({
        features: ['target', 'query', 'gold_generation'],
        num_rows: 8837
    })
    validation: Dataset({
        features: ['target', 'query', 'gold_generation'],
        num_rows: 15757
    })
})

In [ ]:
# Finetune
loss_hist = train(model,
    train_dataloader,
    eval_dataloader,
    batch_size=batch_size,
    max_length=max_length,
    num_epochs=num_epochs,
    lr=lr,
    gradient_accumulation_steps=gradient_accumulation_steps,
    eval_step_num=eval_step_num)

NUMBER of BATCHES PER EPOCH - training: 7879; eval: 1970
cuda:0


/usr/local/lib/python3.10/dist-packages/torch/cuda/memory.py:329: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
  2%|▏         | 132/7879 [00:54<52:51,  2.44it/s]


KeyboardInterrupt: ignored

In [ ]:
plot_history(loss_hist, num_epochs)

NameError: name 'loss_hist' is not defined

In [ ]:
# Empty VRAM
del model
gc.collect()
gc.collect()

0

# Evaluation

## Rouge

### Understand Rouge

In [ ]:
rouge = Rouge()


def calculate_rouge(candidate, reference):
    # candidate, reference: generated and ground-truth sentences
    scores = rouge.get_scores(candidate, reference)[0]['rouge-l']['r'] #rouge-l=statistical measure of the amunt of overlap between strings
                                                                        # 'r' -> recall
    return scores

candidate = "hi, everyone, it's nice to meet you!"
reference = "hi, it's nice to meet everyone!"

# longest subsequences are "hi" and "it's nice to meet"
# 5 words out of 6 in the reference, 5/6 ~ 0.8333333 = rouge-1 (compares single words) -> recall (r)
# 5 words out of 7 in the candidate, 5/7 ~ 0.7142857 = rouge-1 -> precision (p) (how many of the word generated are in the reference text?)
# f = f1-score computed with these values of precision and recall
#https://towardsdatascience.com/introduction-to-text-summarization-with-rouge-scores-84140c64b471
# rouge-2: compares pairs of consecutive words
# rouge-L: measures the Longest Common Subsequence (LCS) words between reference and candidates (words just need to be in sequence, not consecuitve)

print(calculate_rouge(candidate, reference)) #how many words in candidate overlap with the reference's
print(calculate_rouge(reference, candidate))

0.8333333333333334
0.7142857142857143


In [ ]:
# returns average rouge score over set of targets
def calculate_self_rouge(candidates: list[str], references: list[str]):
    rouge_scores = []

    for cand, ref in zip(candidates, references):
        rouge_score = calculate_rouge(cand,ref)
        rouge_scores.append(rouge_score)

    return rouge_scores, np.asarray(rouge_scores).mean()

## Reload Base Model


In [ ]:
# Reload model in FP16 and merge it with LoRA weights
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    low_cpu_mem_usage=True,
    return_dict=True,
    #torch_dtype=torch.float16,
    trust_remote_code=True,
    #device_map=device_map,
)

### Example of base_model prediction

In [ ]:
question, answer = dataset["test"]["query"][0], dataset["test"]["target"][0]
print(f"Question: {question}   \n GT answer: {answer}")

out = tokenizer.decode(base_model.generate(**tokenizer(question, return_tensors="pt").to(base_model.device), max_length=50)[0])

print(f"Predicted answer: {out}")

Question: Who was the man behind The Chipmunks?   
 GT answer: David Seville


Predicted answer: Who was the man behind The Chipmunks?
Answer: The man behind The Chipmunks was a famous chef named Chef Pierre.

3. What did Chef Pierre do to make The Chipmunks a hit?
Answer: Chef


### Evaluate mean rouge

In [ ]:
references = dataset["test"]["target"][:20] #validation
questions = dataset["test"]["query"][:20]
print(questions)
print(references)

['Who was the man behind The Chipmunks?', 'What star sign is Jamie Lee Curtis?', 'Which Lloyd Webber musical premiered in the US on 10th December 1993?', 'Who was the next British Prime Minister after Arthur Balfour?', 'Who had a 70s No 1 hit with Kiss You All Over?', 'What claimed the life of singer Kathleen Ferrier?', 'Rita Coolidge sang the title song for which Bond film?', 'To the nearest million what is the population of Australia?', 'What was the last US state to reintroduce alcohol after prohibition?', 'Which actress was voted Miss Greenwich Village in 1942?', 'What is the Japanese share index called?', "What was the name of Michael Jackson's autobiography written in 1988?", 'In which decade did stereo records first go on sale?', "What was golfing great Ben Hogan's famous reply when he was asked how to improve one's game?", "In what year's Olympics were electric timing devices and a public-address system used for the first time?", "Why is the site of a boxing match called a ring

In [ ]:
out = base_model.generate(**tokenizer(questions, padding=True, return_tensors="pt").to(base_model.device), max_length=50)
candidates = []
for pred in out:
    result = tokenizer.decode(pred, skip_special_tokens=True)
    candidates.append(result)

In [ ]:
candidates

['Who was the man behind The Chipmunks?\n\nAnswer: The man behind The Chipmunks was a famous chef named Gordon Ramsay.\n\nExercise 3: What is the name',
 'What star sign is Jamie Lee Curtis?\nAnswer: Leo\n\nExercise 2: What is the name of the planet that is closest to the sun?\nAnswer: Mercury\n',
 'Which Lloyd Webber musical premiered in the US on 10th December 1993?\n\n(1). The director of the play was more experienced than the stage manager, so the director was able to direct the actors better.',
 'Who was the next British Prime Minister after Arthur Balfour?\n\nAnswer: Clement Attlee\n\nExercise 3: What is the difference between a monarchy and a republic?\n\nAnswer: In',
 'Who had a 70s No 1 hit with Kiss You All Over?\n\nIn a small town called Harmonyville, there lived three friends named Emily, Sarah, and Michael. They were known for their love of',
 'What claimed the life of singer Kathleen Ferrier?\n\nOnce upon a time, in a small town called Harmonyville, there lived two best f

In [ ]:
rouge_scores, mean_rouge_score = calculate_self_rouge(candidates, references)
print(f"Mean Rouge Score for Base Model: {mean_rouge_score}")

Mean Rouge Score for Base Model: 0.0


## Finetuned model

In [ ]:
timestamp = "11_22_2023_17_51_23" #choose model folder
step = 39396 #choose a step
CHECKPOINTS_ROOT_DIR = f"{PROJ_ROOT}/{MODEL_NAME.split('/')[-1]}/{timestamp}/checkpoints"
checkpoint = f"{CHECKPOINTS_ROOT_DIR}/{MODEL_NAME.split('/')[-1]}-finetuned-model-step-{step}"
checkpoint #finetuned weights

'/home/carolina/Anansi/Seminars/LLM_CL/phi-1_5/11_22_2023_10_25_56/checkpoints/phi-1_5-finetuned-model-step-17731'

In [ ]:
model = PeftModel.from_pretrained(base_model, checkpoint)
model = model.merge_and_unload() #we have to merge the orignal with Learned LoRA weights

### Example of finetuned model prediction

In [ ]:
print(f"Question: {question}   \n GT answer: {answer}")

out = tokenizer.decode(model.generate(**tokenizer(question, return_tensors="pt").to(model.device), max_length=50)[0])

print(f"Predicted answer: {out}")

Question: Who was the man behind The Chipmunks?   
 GT answer: David Seville


Predicted answer: Who was the man behind The Chipmunks?John Deacon, the founder of the British National Lottery, was the man behind The Chipmunks. He was born in 1859 in London, England, and was the son of a British


### Evaluate mean rouge

In [ ]:
out_ft = model.generate(**tokenizer(questions, padding=True, return_tensors="pt").to(model.device), max_length=50)
candidates_ft = []
for pred in out_ft:
    result = tokenizer.decode(pred, skip_special_tokens=True)
    candidates_ft.append(result)

In [ ]:
candidates_ft

["Who was the man behind The Chipmunks?Title: The Man Who Took Over the Chipmunks' Home in the Forest, and the Battle of the Chipping Stones in the River",
 'What star sign is Jamie Lee Curtis?Inspector (JLC) is the star sign of a person who is?Aquarius (Aquarius) (or Aquarius',
 'Which Lloyd Webber musical premiered in the US on 10th December 1993?Title Of The Musical In The US On 10th December 1993?The Phantom of The Opera In The US On 10th December 1993?The Phantom',
 'Who was the next British Prime Minister after Arthur Balfour?Title: The Next Prime Minister of Great Britain, after Arthur Balfour, was Edward Heath. He was the first Prime Minister of Great Britain',
 "Who had a 70s No 1 hit with Kiss You All Over?Inner Peace' and 'The Last Man' by Michael Jackson?Jenny and Marko are the characters in the TV series 'The Last",
 "What claimed the life of singer Kathleen Ferrier?In the UK, the Royal College of Physicians has recently published a report on the use of the term 'dementia'

In [ ]:
# Compute Rouge
candidates_ft = tokenizer.decode(model.generate(**tokenizer(questions, padding=True, return_tensors="pt").to(model.device), max_length=50)[0])
rouge_scores_ft, mean_rouge_score_ft = calculate_self_rouge(candidates_ft, references)
print(f"Mean Rouge Score for Finetuned Model: {mean_rouge_score_ft}")

Mean Rouge Score for Finetuned Model: 0.0


# Finetuned model still s****. Why?

1.  Quantization hurst performance.

2. Well, there are a lot of hyperparemeters to be tunned:
*   optimizer
*   lr_scheduler
*   Lora configuration and which modules and layers to finetune.

3. I haven't done any filtering or augmentation to the dataset.
*   Remove corrupted samples (like, incomplete question,...).
*   Don't know exactly what augmentations can be done here: rephrasing questions, provide multiple valid answers to the same question,...

4. There's a memory problem that limits us to train smaller batches or less Lora parameters.

5. I might have picked a difficult dataset to learn, since other "question-answering" datasets like SQUAD focus on predicing the start and end positions of the answer from the given context in the model's input...

# What's up with "outputs.loss"?

"outputs" is a structure, an instance returned by the following subclasses:
https://huggingface.co/docs/transformers/main_classes/output#transformers.modeling_outputs.CausalLMOutput
These subclasses receive as input the model's outputs and loss value.

The code above recreates the loss (CausalLM) used in phi.5 (at least, it gives the same values):

In [ ]:
class MyLoss(nn.Module):
    '''
    https://huggingface.co/learn/nlp-course/chapter7/6?fw=pt#training-with-accelerate
    '''
    def __init__(self) -> None:
        super().__init__()

        self.loss_fct = nn.CrossEntropyLoss(ignore_index=-100, reduction='mean')

    def forward(self, labels: torch.LongTensor, logits: torch.FloatTensor) -> torch.FloatTensor:
        #The input sequence shifted by one to the right forms the labels, since the next token
        # is the label for the current token. We can achieve this by:
        #starting the labels from the second token of the input sequence,
        #since the model does not make a prediction for the first token anyway.
        shift_labels = labels[..., 1:].contiguous()
        # Then we cut off the last logit, as we don’t have a label
        # for the token that follows the full input sequence.
        shift_logits = logits[..., :-1, :].contiguous()

        # Calculate per-token loss
        loss = self.loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        return loss

'''
SIMILAR was found in this link that seems to have the full code for phi1.5:
#https://huggingface.co/microsoft/phi-1_5/raw/a30a931294ac0f344a0c1547877c692ceb17123c/modeling_mixformer_sequential.py
'''